# lxplus + Dask

`LxplusFactory` submits HTCondor batch jobs as Dask workers, each running
inside the Apptainer image you built with [generate_def.ipynb](generate_def.ipynb).

**Prerequisites:**
- `~/worker.sif` built and saved to AFS (see [README.md](README.md))
- Valid VOMS proxy: `voms-proxy-init --voms cms --valid 192:00`
- Run this notebook inside the container: `apptainer exec ~/worker.sif jupyter notebook`

In [ ]:
import sys
sys.path.insert(0, "..")

from coffea_workflow import Step, Workflow, Fileset, Analysis, Plotting, RunConfig, ExecutorConfig, run
from coffea_workflow import facilities
from analysis import get_fileset, run_analysis, plot_results

step_fileset = Step(name="Fileset", step_type=Fileset, builder=get_fileset)
step_analysis = Step(name="Analysis", step_type=Analysis, builder=run_analysis)
step_plotting = Step(name="Plotting", step_type=Plotting, builder=plot_results)

workflow = Workflow()
workflow.add(step_fileset)
workflow.add(step_analysis, depends_on=[step_fileset])
workflow.add(step_plotting, depends_on=[step_analysis])

config = RunConfig(
    strategy="by_dataset",
    cache_dir=".cache_lxplus",
    facility=facilities.LxplusFactory(
        worker_image="~/worker.sif",
        queue="longlunch",   # espresso=20min, longlunch=2h, workday=8h ...
        workers=10,
    ),
    executor_config=ExecutorConfig(executor_type="DaskExecutor"),
)

run(workflow, config)